# 01 - Explore MOBPRO commuting records

**Source:** `FD_MOBPRO_2023.csv`. **Default scope: first 10,000 rows only.**
**Module:** `scripts/datasets/mobpro.py`.

Start with a sample to understand columns and values. First rows are not random or
representative of France or Ile-de-France. Weighted sample summaries describe only those
loaded records. Main mode does not reveal every leg of a journey. 


In [1]:
import sys
from pathlib import Path
import pandas as pd
import plotly.io as pio
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError('Open the hackathon project folder before running this notebook.')
if not Path(sys.executable).resolve().is_relative_to(ROOT / '.venv'):
    raise RuntimeError('Select the project .venv kernel, then restart and run all.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.datasets import mobpro
from scripts.exploration import column_summary, repeated_keys, save_parquet

pio.renderers.default = 'plotly_mimetype'
print('Python:', sys.executable)


Python: d:\Hackathon Mobilités 2026 Spécial Dataviz\.venv\Scripts\python.exe


In [9]:
source = ROOT / mobpro.FILENAME
SAMPLE_ROWS = 10_000
raw = mobpro.load_sample(source, rows=SAMPLE_ROWS)
print('Explore: first', SAMPLE_ROWS, 'source rows; not a representative population sample.')


Explore: first 10000 source rows; not a representative population sample.


In [10]:
print('Loaded rows and columns -- shape:', raw.shape)
display(raw.head(10))
raw.info()


Loaded rows and columns -- shape: (10000, 32)


,COMMUNE,ARM,DCFLT,DCLT,AGEREVQ,GS,DEROU,DIPL,EMPL,ILT,...,REGION,REGLT,SEXE,STAT,STOCD,TP,TRANS,TYPL,TYPMR,VOIT
0,01001,ZZZZZ,99999,01001,025,5,Z,105,101,1,...,84,84,2,111,10,11,1,1,41,2
1,01001,ZZZZZ,99999,01001,020,2,Z,105,ZZZ,1,...,84,84,2,121,10,11,1,1,41,3
2,01001,ZZZZZ,99999,01001,035,4,Z,109,101,1,...,84,84,2,111,10,11,5,1,41,2
3,01001,ZZZZZ,99999,01001,040,2,Z,108,ZZZ,1,...,84,84,2,121,21,11,1,1,41,2
4,01001,ZZZZZ,99999,01001,055,1,Z,106,ZZZ,1,...,84,84,1,121,10,11,5,1,41,2
5,01001,ZZZZZ,99999,01001,045,5,Z,105,101,1,...,84,84,2,111,22,12,5,2,12,1
6,01001,ZZZZZ,99999,01001,055,2,Z,104,ZZZ,1,...,84,84,1,121,10,11,5,1,41,1
7,01001,ZZZZZ,99999,01001,055,2,Z,101,ZZZ,1,...,84,84,2,121,10,11,1,1,41,1
8,01001,ZZZZZ,99999,01001,030,6,Z,106,101,1,...,84,84,1,111,10,11,5,1,41,2
9,01001,ZZZZZ,99999,01001,030,5,Z,105,101,1,...,84,84,2,111,10,11,5,1,41,2


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 32 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   COMMUNE   10000 non-null  string
 1   ARM       10000 non-null  string
 2   DCFLT     10000 non-null  string
 3   DCLT      10000 non-null  string
 4   AGEREVQ   10000 non-null  string
 5   GS        10000 non-null  string
 6   DEROU     10000 non-null  string
 7   DIPL      10000 non-null  string
 8   EMPL      10000 non-null  string
 9   ILT       10000 non-null  string
 10  ILTUU     10000 non-null  string
 11  IMMI      10000 non-null  string
 12  INATC     10000 non-null  string
 13  INEEM     10000 non-null  string
 14  INPOM     10000 non-null  string
 15  INPSM     10000 non-null  string
 16  IPONDI    10000 non-null  string
 17  LPRM      10000 non-null  string
 18  METRODOM  10000 non-null  string
 19  MOCO      10000 non-null  string
 20  NA5       10000 non-null  string
 21  NPERR     10000 non-null

## Clean a copy and inspect the columns

The original file and `raw` table stay unchanged. Strip surrounding spaces and turn
blank cells into missing values; preserve special codes such as `NA` and `ZZ`.
Dataset-specific numeric conversions are documented in the module's `clean()` function.
Unexpected numeric text raises an error for inspection. No rows are dropped or deduplicated.


In [4]:
cleaned = mobpro.clean(raw)
assert len(cleaned) == len(raw)
display(cleaned.head())
display(column_summary(cleaned))


,COMMUNE,ARM,DCFLT,DCLT,AGEREVQ,GS,DEROU,DIPL,EMPL,ILT,...,REGLT,SEXE,STAT,STOCD,TP,TRANS,TYPL,TYPMR,VOIT,weight_is_valid
0,01001,ZZZZZ,99999,01001,025,5,Z,105,101,1,...,84,2,111,10,11,1,1,41,2,True
1,01001,ZZZZZ,99999,01001,020,2,Z,105,ZZZ,1,...,84,2,121,10,11,1,1,41,3,True
2,01001,ZZZZZ,99999,01001,035,4,Z,109,101,1,...,84,2,111,10,11,5,1,41,2,True
3,01001,ZZZZZ,99999,01001,040,2,Z,108,ZZZ,1,...,84,2,121,21,11,1,1,41,2,True
4,01001,ZZZZZ,99999,01001,055,1,Z,106,ZZZ,1,...,84,1,121,10,11,5,1,41,2,True


,data_type,missing_rows,distinct_values
column,,,
COMMUNE,string,0,42
ARM,string,0,1
DCFLT,string,0,51
DCLT,string,0,655
AGEREVQ,string,0,17
GS,string,0,6
DEROU,string,0,2
DIPL,string,0,10
EMPL,string,0,3


## Geography columns: home vs work

| Column | Meaning |
|---|---|
| `COMMUNE` | Residence INSEE commune code (home) |
| `DCLT` | Workplace code; includes municipal arrondissements for Paris, Lyon and Marseille |
| `REGION` | Residence region code (home) |
| `REGLT` | Workplace region code (work) |

**Remember:** `REGION = "11"` selects people living in Île-de-France; `REGLT = "11"` selects people working there. Keep codes as text: INSEE codes are not postal codes.


## Inspect weights and categories

`IPONDI` becomes numeric. `weight_is_valid` marks positive, finite, present weights.
Investigate invalid rows before explicitly deciding whether to exclude them. Geography
and mode codes remain strings. No Paris-arrondissement grouping is applied.


In [5]:
display(cleaned['IPONDI'].describe())
display(cleaned.loc[~cleaned.weight_is_valid])
display(cleaned['REGLT'].value_counts(dropna=False).rename('sample_records'))
display(cleaned['TRANS'].value_counts(dropna=False).rename('sample_records'))


count     10000.0
mean     4.033182
std      1.307335
min      0.993316
25%      3.440905
50%      4.628721
75%      4.982609
max      6.144448
Name: IPONDI, dtype: Float64

,COMMUNE,ARM,DCFLT,DCLT,AGEREVQ,GS,DEROU,DIPL,EMPL,ILT,...,REGLT,SEXE,STAT,STOCD,TP,TRANS,TYPL,TYPMR,VOIT,weight_is_valid


REGLT
84    8715
99     854
27     302
11      61
93      19
44      16
24      10
32      10
75       8
06       1
76       1
ZZ       1
52       1
28       1
Name: sample_records, dtype: int64[pyarrow]

TRANS
5    7754
6    1018
2     570
1     391
3     163
4     104
Name: sample_records, dtype: int64[pyarrow]

In [6]:
mode_summary = mobpro.mode_summary(cleaned)
display(mode_summary)
mobpro.plot_modes(mode_summary, scope=f'first {len(cleaned):,} source records only').show()


,TRANS,sample_records,estimated_workers,share_pct
0,1,391,1631.942447,4.04629
1,2,570,2255.410446,5.592137
2,3,163,675.028403,1.673687
3,4,104,458.169117,1.135999
4,5,7754,31576.789334,78.292502
5,6,1018,3734.478197,9.259385


## Optional: scan the full source in chunks

Turn this on when ready. Region `11` selects workplaces in Ile-de-France, including
residents of other regions. Set `WORKPLACE_REGION = None` to include all workplace regions.
Each chunk is reduced to a small mode summary. No other dataset is loaded.

Estimates use `sum(IPONDI)`, not daily trip counts. Mode `1` (no transport) remains in the
denominator. Inspect unknown mode codes against the dictionary before interpreting results.


In [7]:
RUN_FULL_SCAN = False
WORKPLACE_REGION = '11'

if RUN_FULL_SCAN:
    parts = []
    for chunk in mobpro.read_chunks(source, columns=['TRANS', 'IPONDI', 'REGLT']):
        chunk = mobpro.clean(chunk)
        if WORKPLACE_REGION is not None:
            chunk = chunk.loc[chunk.REGLT.eq(WORKPLACE_REGION)]
        parts.append(mobpro.mode_summary(chunk))

    full_mode_summary = pd.concat(parts).groupby('TRANS', dropna=False)[
        ['sample_records', 'estimated_workers']
    ].sum().reset_index()
    full_mode_summary['share_pct'] = (
        full_mode_summary.estimated_workers / full_mode_summary.estimated_workers.sum() * 100
    )
    display(full_mode_summary)
else:
    print('Full scan is off. The preceding results describe the loaded sample only.')


Full scan is off. The preceding results describe the loaded sample only.


## Your exploration notes

Record the scope, variable definitions, unexpected values and remaining questions.
The mode-summary table gives the values behind the chart. High observed car use is not
by itself evidence of car dependence.


## Optional: save this dataset as Parquet



In [8]:
SAVE_CLEANED = False
output_path = ROOT / 'data' / 'cleaned' / 'mobpro_sample.parquet'

if SAVE_CLEANED:
    save_parquet(cleaned, output_path)
    print('Saved', len(cleaned), 'rows to', output_path)
else:
    print('No cleaned data written. Change SAVE_CLEANED after reviewing this dataset.')


No cleaned data written. Change SAVE_CLEANED after reviewing this dataset.
